<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/tapta-data-scalability/notebooks/tapta_p4_diagnostics_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 MASTER DIAGNOSTICS: TAPTA P4 Performance Diagnosis (D01 - D09)
### Đề tài: Continual Relation Extraction with Task-Aware Prompt Adaptation

Notebook này phục vụ kế hoạch chẩn đoán khoa học toàn diện **TAPTA Performance Diagnosis & Research Recovery Plan** (Phase 1):
- **D01:** Oracle Task + Task-Local Classifier (Trần phân biệt nội bộ quan hệ trong task)
- **D02:** Oracle Task + Global Classifier (Mức độ can nhiễu cross-task khi biết trước task)
- **D03:** Oracle Top-K Routing (Hiệu năng nếu Router không bao giờ miss task đúng)
- **D04:** Hard Top-1 Routing ($K=1$)
- **D05:** Reference Top-2 Soft Fusion ($K=2$, P4 Chuẩn)
- **D06:** Top-4 Soft Fusion ($K=4$)
- **D07:** All-Task Fusion ($K=8$)
- **D08:** Score Calibration (Hiệu chỉnh điểm số Cosine Z-score trước khi Fusion)
- **D09:** Frozen BERT + Flat Nearest Prototype Baseline (Baseline unprompted $M=0$)
- **Error Taxonomy:** Xuất chi tiết `predictions.jsonl` phục vụ phân tích nguyên nhân lỗi sample-level.

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Check)

In [ ]:
import torch
print("=" * 60)
print("CUDA / GPU PREFLIGHT CHECK")
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:   {torch.version.cuda}")
else:
    print("⚠️ CẢNH BÁO: Không tìm thấy GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU.")
print("=" * 60)

## 2. Clone / Cập Nhật Repository `research-code-lab`

In [ ]:
import os
REPO_URL = "https://github.com/vinhlq2512/research-code-lab.git"
BRANCH = "feat/tapta-data-scalability"
REPO_DIR = "/content/research-code-lab"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scikit-learn tabulate

## 3. Thực Nghiệm Chẩn Đoán 1: TACRED Track A (D01 - D09)
*(40 quan hệ, 8 tác vụ, strict 5-shot, Rehearsal-Free $M=0$, seed 2021)*

In [ ]:
!python experiments/diagnostics/run_p4_diagnostics.py \
    --config configs/p4_hierarchical_prompt_tree_tacred_5shot_seed2021.yaml \
    --device cuda \
    --output_dir runs/diagnostics/tacred_5shot/seed_2021

## 4. Thực Nghiệm Chẩn Đoán 2: FewRel Track A (D01 - D09)
*(80 quan hệ, 8 tác vụ, strict 5-shot, Rehearsal-Free $M=0$, seed 2021)*

In [ ]:
!python experiments/diagnostics/run_p4_diagnostics.py \
    --config configs/p4_hierarchical_prompt_tree_fewrel_5shot_seed2021.yaml \
    --device cuda \
    --output_dir runs/diagnostics/fewrel_5shot/seed_2021

## 5. Tổng Hợp & Đóng Gói Toàn Bộ Kết Quả Chẩn Đoán (Zip Package)

In [ ]:
from google.colab import files
import shutil

print("Đang nén kết quả chẩn đoán D01 - D09...")
shutil.make_archive("/content/tapta_p4_diagnostics_results", "zip", "/content/research-code-lab/runs/diagnostics")

print("Tự động tải về tệp tapta_p4_diagnostics_results.zip...")
files.download("/content/tapta_p4_diagnostics_results.zip")